<a href="https://colab.research.google.com/github/lucas-j-zheng/ml-practice-from-scratch/blob/main/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import RobertaModel, RobertaTokenizerFast, AutoTokenizer
from sklearn.metrics import accuracy_score, matthews_corrcoef
from scipy.stats import pearsonr
import random
from datasets import load_dataset


import math


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
def set_seed(s=0):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(s)

TASK_CONFIG = {
    'mrpc':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 2, 'is_regression': False},
    'cola':  {'cols': ('sentence',  None),         'num_outputs': 2, 'is_regression': False},
    'stsb':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 1, 'is_regression': True},
}

In [ ]:
MODEL_NAME = 'roberta-base'
HIDDEN = 768

## Data Loading



In [ ]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
class GLUEDataset(torch.utils.data.Dataset):
  def __init__(self, texts_a, texts_b, labels, tokenizer, is_regression):
    self.texts_a = texts_a
    self.texts_b = texts_b
    self.labels = labels
    self.tokenizer = tokenizer
    self.is_regression = is_regression

  def __len__(self):
    return len(self.labels)


  def __getitem__(self, i):
    # tokenize and return one example
    if self.texts_b is None:
      encoded = self.tokenizer(
        self.texts_a[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    else:
      encoded = self.tokenizer(
        self.texts_a[i], self.texts_b[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    label_dtype = torch.float if self.is_regression else torch.long
    label = torch.tensor(self.labels[i], dtype=label_dtype)

    # squeeze for additional dimension from turning into pt (batch dim)
    return {
        'input_ids': encoded['input_ids'].squeeze(0),
        'attention_mask': encoded['attention_mask'].squeeze(0),
        'labels': label,
    }





In [ ]:

def build_dataloaders(task, batch_size=32):
  cfg = TASK_CONFIG[task]
  col_a, col_b = cfg['cols']
  is_regression = cfg['is_regression']
  raw = load_dataset('glue', task)

  def make_split(split):
    a = raw[split][col_a]
    b = raw[split][col_b] if col_b is not None else None
    y = raw[split]['label']
    return GLUEDataset(a, b, y, tokenizer, is_regression)

  train = make_split('train')
  val = make_split('validation')
  test = make_split('test')

  train_dl = DataLoader(train, batch_size=batch_size, shuffle=True)
  val_dl = DataLoader(val, batch_size=batch_size, shuffle=False)

  return train_dl, val_dl

# Model Finetuning (FULL)

In [ ]:
class RobertaClassifier(nn.Module):
  def __init__(self, base, num_outputs):
    super().__init__()
    self.base = base
    self.dropout = nn.Dropout(0.1)
    self.head = nn.Linear(HIDDEN, num_outputs)

  def forward(self, input_ids, attention_mask):
    # forward pass through the base model
    out = self.base(input_ids=input_ids, attention_mask=attention_mask)

    # Take the position 0 <s> token
    CLS = out.last_hidden_state[:, 0, :]

    # Apply dropout and classification head
    x = self.dropout(CLS)
    logits = self.head(x)
    return logits


Testing Roberta Classifier


In [ ]:
base = RobertaModel.from_pretrained('roberta-base')
model = RobertaClassifier(base, num_outputs=2)

ids = torch.randint(0, 1000, (4, 128))
mask = torch.ones_like(ids)
out = model(ids, mask)
print(out.shape)   # expect (4, 2) due to (4, num_outputs)
print(out.dtype)   # expect torch.float32

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([4, 2])
torch.float32


In [ ]:
def evaluate(model, dataloader, task, device):
  model.eval()
  cfg = TASK_CONFIG[task]
  is_regression = cfg['is_regression']
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch in dataloader:
      batch = {k: v.to(device) for k,v in batch.items()}
      logits = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'])

      if is_regression:
        preds = logits.squeeze(-1)

      else:
        preds = logits.argmax(dim = -1)

      all_preds.append(preds.cpu().numpy())
      all_labels.append(batch['labels'].cpu().numpy())

  all_preds = np.concatenate(all_preds)
  all_labels = np.concatenate(all_labels)

  if task == 'mrpc':
    return accuracy_score(all_labels, all_preds)
  elif task == 'cola':
    return matthews_corrcoef(all_labels, all_preds)
  elif task == 'stsb':
    return pearsonr(all_preds, all_labels)[0]




In [ ]:
results = {}
for task in TASK_CONFIG.keys():
    print(f"\n{'='*40}")
    print(f"Starting Fine-Tuning for: {task.upper()}")
    print(f"{'='*40}")
    best_val = train(task, use_lora=False, epochs=3, lr=2e-5, batch_size=16)
    results[task] = best_val
    print(f'\n{task.upper()} FT best val: {best_val:.4f}\n')

print("\n--- Final Summary ---")
for task, val in results.items():
    print(f"{task.upper()}: {val:.4f}")


Starting Fine-Tuning for: MRPC


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 | Step 0 | Batch loss: 8.8142
  Epoch 1 | Step 20 | Batch loss: 1.7270
  Epoch 1 | Step 40 | Batch loss: 1.8134
  Epoch 1 | Step 60 | Batch loss: 1.6611
  Epoch 1 | Step 80 | Batch loss: 0.6921
  Epoch 1 | Step 100 | Batch loss: 0.8808
  Epoch 1 | Step 120 | Batch loss: 0.6105
  Epoch 1 | Step 140 | Batch loss: 0.6958
  Epoch 1 | Step 160 | Batch loss: 1.3626
  Epoch 1 | Step 180 | Batch loss: 0.6401
  Epoch 1 | Step 200 | Batch loss: 0.9206
  Epoch 1 | Step 220 | Batch loss: 0.5902
  Epoch 1 | Step 240 | Batch loss: 0.4794
  Epoch 1 | Step 260 | Batch loss: 0.4782
  Epoch 1 | Step 280 | Batch loss: 0.5343
  Epoch 1 | Step 300 | Batch loss: 0.3004
  Epoch 1 | Step 320 | Batch loss: 0.4383
  Epoch 1 | Step 340 | Batch loss: 0.3330
  epoch 1: avg_train_loss=1.0160, val=0.8807
  Epoch 2 | Step 0 | Batch loss: 0.3776
  Epoch 2 | Step 20 | Batch loss: 0.3020
  Epoch 2 | Step 40 | Batch loss: 0.4426
  Epoch 2 | Step 60 | Batch loss: 0.4004
  Epoch 2 | Step 80 | Batch loss: 0.5573
 

In [ ]:
def train_diag(task='mrpc', epochs=3, lr=2e-5, batch_size=16):
    set_seed(42)
    train_dl, val_dl = build_dataloaders(task, batch_size)
    cfg = TASK_CONFIG[task]

    base = RobertaModel.from_pretrained('roberta-base')
    model = RobertaClassifier(base, num_outputs=cfg['num_outputs']).to(DEVICE)

    # Diagnostic 1: count trainable params
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'trainable params: {n_train:,}')

    # Diagnostic 2: pre-training val
    pre_val = evaluate(model, val_dl, task, DEVICE)
    print(f'pre-training val: {pre_val:.4f}')

    # Diagnostic 3: head norm before training
    print(f'head norm pre-train:  {model.head.weight.norm().item():.4f}')
    base_param_pre = next(iter(model.base.parameters())).clone().detach()

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01
    )

    model.train()
    for step, batch in enumerate(train_dl):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(batch['input_ids'], batch['attention_mask'])
        loss = loss_fn(logits, batch['labels'])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            head_norm = model.head.weight.norm().item()
            base_now = next(iter(model.base.parameters()))
            base_diff = (base_now - base_param_pre).abs().max().item()
            print(f'  step {step}: loss={loss.item():.4f} head_norm={head_norm:.4f} base_changed_by={base_diff:.6f}')

    print(f'head norm post-train: {model.head.weight.norm().item():.4f}')

    # Diagnostic 4: post-training val + prediction distribution
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(batch['input_ids'], batch['attention_mask'])
            preds = logits.argmax(dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch['labels'].cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    print(f'\nPRED distribution:  {np.bincount(all_preds)}')
    print(f'LABEL distribution: {np.bincount(all_labels)}')
    print(f'val accuracy: {(all_preds == all_labels).mean():.4f}')

train_diag()

# LORA

In [ ]:
class LoraLinear(nn.Module):
  def __init__(self, base, r=8, alpha=16):
    super().__init__()
    self.base = base
    for p in self.base.parameters():
      p.requires_grad = False

    self.r = r
    self.alpha = alpha
    self.scaling = alpha/r

    in_features = base.in_features
    out_features = base.out_features

    # 4. Create LoRA matrices: A nonzero, B zero
    self.A = nn.Parameter(torch.empty(r, in_features))
    self.B = nn.Parameter(torch.zeros(out_features, r))
    nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))


  def forward(self, x):
    base_out = self.base(x)
    lora_out = (x @ self.A.T) @ self.B.T
    return base_out + self.scaling * lora_out


In [ ]:
def apply_lora_to_qv(base, r=8, alpha=16):
  for layer in base.encoder.layer:
    attn = layer.attention.self
    attn.query = LoraLinear(attn.query, r=r, alpha=alpha)
    attn.value = LoraLinear(attn.value, r=r, alpha=alpha)


In [ ]:
base = RobertaModel.from_pretrained(MODEL_NAME)
apply_lora_to_qv(base, r=8, alpha=16)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
attn = base.encoder.layer[0].attention.self
print('query:', type(attn.query).__name__)   # LoraLinear
print('key:  ', type(attn.key).__name__)     # Linear
print('value:', type(attn.value).__name__)   # LoraLinear

# Model still runs end-to-end
ids = torch.randint(0, 1000, (2, 16))
mask = torch.ones_like(ids)
out = base(input_ids=ids, attention_mask=mask)
print('output shape:', out.last_hidden_state.shape)   # (2, 16, 768)

query: LoraLinear
key:   Linear
value: LoraLinear
output shape: torch.Size([2, 16, 768])


In [ ]:
# Param count check: 12 layers × 2 wrappers × 2*r*768
trainable_lora = sum(p.numel() for n, p in base.named_parameters()
                     if p.requires_grad and ('A' in n or 'B' in n))
print(f'LoRA trainable params: {trainable_lora:,}')   # 294,912

LoRA trainable params: 294,912


In [ ]:
def freeze_for_lora(model):
  for name, param in model.named_parameters():
    is_lora = name.endswith('.A') or name.endswith('.B')
    is_head = name.startswith('head.')
    param.requires_grad = is_lora or is_head

In [ ]:
base = RobertaModel.from_pretrained(MODEL_NAME)
apply_lora_to_qv(base, r=8, alpha=16)
model = RobertaClassifier(base, num_outputs=2)
freeze_for_lora(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable:,} / {total:,} total')

# show what's trainable
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f'  {name} {tuple(p.shape)}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable: 296,450 / 124,942,082 total
  base.encoder.layer.0.attention.self.query.A (8, 768)
  base.encoder.layer.0.attention.self.query.B (768, 8)
  base.encoder.layer.0.attention.self.value.A (8, 768)
  base.encoder.layer.0.attention.self.value.B (768, 8)
  base.encoder.layer.1.attention.self.query.A (8, 768)
  base.encoder.layer.1.attention.self.query.B (768, 8)
  base.encoder.layer.1.attention.self.value.A (8, 768)
  base.encoder.layer.1.attention.self.value.B (768, 8)
  base.encoder.layer.2.attention.self.query.A (8, 768)
  base.encoder.layer.2.attention.self.query.B (768, 8)
  base.encoder.layer.2.attention.self.value.A (8, 768)
  base.encoder.layer.2.attention.self.value.B (768, 8)
  base.encoder.layer.3.attention.self.query.A (8, 768)
  base.encoder.layer.3.attention.self.query.B (768, 8)
  base.encoder.layer.3.attention.self.value.A (8, 768)
  base.encoder.layer.3.attention.self.value.B (768, 8)
  base.encoder.layer.4.attention.self.query.A (8, 768)
  base.encoder.layer.4.att

In [ ]:
from transformers import get_linear_schedule_with_warmup

def train(task, use_lora=False, epochs=3, lr=2e-5, batch_size=32, r=8, alpha=16):
  #set seed first
  set_seed(0)
  train_dl, val_dl = build_dataloaders(task, batch_size)
  cfg = TASK_CONFIG[task]
  is_regression = cfg['is_regression']

  base = RobertaModel.from_pretrained('roberta-base')
  if use_lora:
    apply_lora_to_qv(base, r=r, alpha=alpha)

  model = RobertaClassifier(base, num_outputs=cfg['num_outputs'])

  if use_lora:
    freeze_for_lora(model)


  model.to(DEVICE)

  #need loss

  loss_fn = nn.MSELoss() if is_regression else nn.CrossEntropyLoss()

  #only optimizer for trainable params!

  optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=lr, weight_decay=0.01
                                )

  # total_steps = len(train_dl) * epochs
  # scheduler = get_linear_schedule_with_warmup(
  #     optimizer,
  #     num_warmup_steps=int(0.1 * total_steps),
  #     num_training_steps=total_steps
  # )

  best = -1e9

  for epoch in range(epochs):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_dl):
      # a. move to device
      batch = {k: v.to(DEVICE) for k, v in batch.items()}

      # b. forward
      logits = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'])

      # c. compute loss (branch on regression)
      if is_regression:
        loss = loss_fn(logits.squeeze(-1), batch['labels'])
      else:
        loss = loss_fn(logits, batch['labels'])

      # d. zero grads
      optimizer.zero_grad()

      # e. backward
      loss.backward()

      # f. step
      optimizer.step()
      # scheduler.step()
      total_loss += loss.item()

      if step % 20 == 0:
        print(f'  Epoch {epoch+1} | Step {step} | Batch loss: {loss.item():.4f}')

    avg_train_loss = total_loss / len(train_dl)
    val_metric = evaluate(model, val_dl, task, DEVICE)
    best = max(best, val_metric)
    print(f'  epoch {epoch+1}: avg_train_loss={avg_train_loss:.4f}, val={val_metric:.4f}')

  return best


In [ ]:
best_lora_mrpc = train(
    'mrpc',
    use_lora=True,
    epochs=30,
    lr=4e-4,
    batch_size=16,
    r=8,
    alpha=8,
)
print(f'\nLoRA MRPC best val: {best_lora_mrpc:.4f}')

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 | Step 0 | Batch loss: 0.6624
  Epoch 1 | Step 20 | Batch loss: 0.5326
  Epoch 1 | Step 40 | Batch loss: 0.6035
  Epoch 1 | Step 60 | Batch loss: 0.5771
  Epoch 1 | Step 80 | Batch loss: 0.6000
  Epoch 1 | Step 100 | Batch loss: 0.4557
  Epoch 1 | Step 120 | Batch loss: 0.6161
  Epoch 1 | Step 140 | Batch loss: 0.3676
  Epoch 1 | Step 160 | Batch loss: 0.5739
  Epoch 1 | Step 180 | Batch loss: 0.4470
  Epoch 1 | Step 200 | Batch loss: 0.5242
  Epoch 1 | Step 220 | Batch loss: 0.3837
  epoch 1: avg_train_loss=0.5590, val=0.7966
  Epoch 2 | Step 0 | Batch loss: 0.7194
  Epoch 2 | Step 20 | Batch loss: 0.2725
  Epoch 2 | Step 40 | Batch loss: 0.3040
  Epoch 2 | Step 60 | Batch loss: 0.4050
  Epoch 2 | Step 80 | Batch loss: 0.3751
  Epoch 2 | Step 100 | Batch loss: 0.2674
  Epoch 2 | Step 120 | Batch loss: 0.5069
  Epoch 2 | Step 140 | Batch loss: 0.4072
  Epoch 2 | Step 160 | Batch loss: 0.3506
  Epoch 2 | Step 180 | Batch loss: 0.2700
  Epoch 2 | Step 200 | Batch loss: 0.6591
 

In [ ]:
print('=== CoLA LoRA ===')
best_lora_cola = train('cola', use_lora=True, epochs=20, lr=4e-4, batch_size=32, r=8, alpha=8)

print('\n=== STS-B LoRA ===')
best_lora_stsb = train('stsb', use_lora=True, epochs=20, lr=4e-4, batch_size=16, r=8, alpha=8)

=== CoLA LoRA ===


cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 | Step 0 | Batch loss: 7.3145
  Epoch 1 | Step 20 | Batch loss: 4.0742
  Epoch 1 | Step 40 | Batch loss: 4.0159
  Epoch 1 | Step 60 | Batch loss: 2.6135
  Epoch 1 | Step 80 | Batch loss: 1.4381
  Epoch 1 | Step 100 | Batch loss: 1.7867
  Epoch 1 | Step 120 | Batch loss: 0.7540
  Epoch 1 | Step 140 | Batch loss: 0.8628
  Epoch 1 | Step 160 | Batch loss: 1.2650
  Epoch 1 | Step 180 | Batch loss: 1.0220
  Epoch 1 | Step 200 | Batch loss: 0.6422
  Epoch 1 | Step 220 | Batch loss: 1.2023
  Epoch 1 | Step 240 | Batch loss: 0.5221
  Epoch 1 | Step 260 | Batch loss: 0.4981
  Epoch 1 | Step 280 | Batch loss: 0.8959
  Epoch 1 | Step 300 | Batch loss: 0.9826
  Epoch 1 | Step 320 | Batch loss: 0.6851
  Epoch 1 | Step 340 | Batch loss: 0.7519
  epoch 1: avg_train_loss=1.4738, val=0.8708
  Epoch 2 | Step 0 | Batch loss: 0.6197
  Epoch 2 | Step 20 | Batch loss: 0.7126
  Epoch 2 | Step 40 | Batch loss: 0.3006
  Epoch 2 | Step 60 | Batch loss: 0.4068
  Epoch 2 | Step 80 | Batch loss: 1.3917
 